In [4]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

In [5]:
df = pd.read_csv("../data/processed/processed_logs.csv")

print(df.shape)

df.head()

(4945, 23)


,Raw_Log,Component,Line_Number,Timestamp,Severity,Username,AccountID,Originating_Address,Destination_Address,TPS,...,PositiveResponse,QueueInserted,RouterSent,ContainsError,Message_Length,Word_Count,Hour,Minute,Second,Millisecond
0,[ESME.cpp|01321|02:46:17:014|~CR~][CBS_Product...,ESME.cpp,1321,1900-01-01 02:46:17.014,~CR~,0,0.0,0,0,0.0,...,1,0,0,0,99,3,2,46,17,14
1,[ESME.cpp|00448|02:46:17:059|~CR~][SimulQ2.824...,ESME.cpp,448,1900-01-01 02:46:17.059,~CR~,0,0.0,0,0,0.0,...,0,1,0,0,116,6,2,46,17,59
2,[Globals.cpp|00107|02:46:17:059|~CR~]Checkign ...,Globals.cpp,107,1900-01-01 02:46:17.059,~CR~,SimulQ2,0.0,Yas,0,0.0,...,0,0,0,0,73,3,2,46,17,59
3,[Globals.cpp|00124|02:46:17:060|~CR~]Checkign ...,Globals.cpp,124,1900-01-01 02:46:17.060,~CR~,0,0.0,0,255653530543,0.0,...,0,0,0,0,77,3,2,46,17,60
4,[ESME.cpp|04785|02:46:17:060|~CR~]A2P Authenti...,ESME.cpp,4785,1900-01-01 02:46:17.060,~CR~,0,0.0,0,0,0.0,...,0,0,0,0,60,3,2,46,17,60


In [6]:
df["Raw_Log"].head()


0    [ESME.cpp|01321|02:46:17:014|~CR~][CBS_Product...
1    [ESME.cpp|00448|02:46:17:059|~CR~][SimulQ2.824...
2    [Globals.cpp|00107|02:46:17:059|~CR~]Checkign ...
3    [Globals.cpp|00124|02:46:17:060|~CR~]Checkign ...
4    [ESME.cpp|04785|02:46:17:060|~CR~]A2P Authenti...
Name: Raw_Log, dtype: str

In [7]:
!pip install transformers
!pip install torch
!pip install sentencepiece


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [8]:
import torch
from transformers import AutoTokenizer
from transformers import AutoModel

Load pretrained LogBERT

In [9]:
MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModel.from_pretrained(MODEL_NAME)

model.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=

Generate embeddings

In [10]:
# ----------------------------
# Generate Embeddings (Chunking)
# ----------------------------

import torch

# Use Apple GPU if available
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = model.to(device)

print(f"Using device: {device}")

chunk_size = 32
embeddings = []

for i in tqdm(range(0, len(df), chunk_size), desc="Generating Embeddings"):

    chunk = (
        df["Raw_Log"]
        .iloc[i:i + chunk_size]
        .fillna("")
        .astype(str)
        .tolist()
    )

    inputs = tokenizer(
        chunk,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    )

    # Move tensors to device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    # CLS token embeddings
    batch_embeddings = outputs.last_hidden_state[:, 0, :]

    embeddings.append(batch_embeddings.cpu().numpy())

# Merge all batches
embeddings = np.vstack(embeddings)

print("Embeddings Shape:", embeddings.shape)

Using device: mps


Generating Embeddings: 100%|██████████| 155/155 [01:01<00:00,  2.50it/s]

Embeddings Shape: (4945, 768)
